## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])

 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = userdata.get('newapi')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f"Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\nFinal Summary:\n", result["summary"])

Searching: Quantum Computing
Analyzing 2 results
Searching: Quantum Computing
Analyzing 4 results
Generating summary...

Final Summary:
 Here's a summary of the analysis in simple terms:

**The Power of Quantum Computing**

Quantum computers have the potential to solve complex problems much faster than regular computers. They can process huge amounts of data in a unique way called superposition, which makes them incredibly powerful.

**Benefits and Applications**

Quantum computing can be used in various fields such as:

- **Cryptography and Cybersecurity**: Secure online communication and prevent hacking
- **Optimization**: Solve complex problems in logistics, finance, and energy management
- **Materials Science**: Discover new materials and substances with unique properties
- **Personalized Medicine**: Understand the genetic roots of diseases and create targeted treatments
- **Climate Modeling**: Simulate chemical reactions to develop new, sustainable materials and fuels

**Challenge

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('newapi')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'GROQ_API_KEY'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",   # Example Groq model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

Bot says: I'm a large language model, I don't have a physical store. I exist solely as a digital entity, so I don't have store hours or a physical location. I'm available 24/7, and you can interact with me through text-based conversations at any time. How can I assist you today?
